In [0]:
from pyspark.sql.functions import from_json, current_timestamp
     

In [0]:

# 1. Инициализация виджетов

dbutils.widgets.text("evh_name", "artemzharkov10_evh")
dbutils.widgets.text("catalog", "dbr_dev")
dbutils.widgets.text("schema", "artemzharkov10_bronze")

EVH_NAME = dbutils.widgets.get("evh_name")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
  

In [0]:
   
# 2. Настройка конфигурации подключения Kafka / Event Hub
EH_CONN_STR = dbutils.secrets.get(scope="default2", key="artem-evh-connector")

BOOTSTRAP = "evhpl24databricks.servicebus.windows.net:9093"
JAAS = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{EH_CONN_STR}";'

# 3. Определение путей и целевой таблицы
checkpoint_path = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints/weather_stream_checkpoint"
bronze_table_name = f"{CATALOG}.{SCHEMA}.bronze_streaming_weather"
     


In [0]:
# 4. Чтение потока (creating a query with JAAS: like a passport to Azure Event Hub)
raw_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", EVH_NAME) 
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", JAAS) # passport to enter in evh
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", 100) # batch size
    .load()
)


In [0]:
landing_df = (
    raw_df.selectExpr("CAST(value AS STRING) as json_payload", "timestamp as eventhub_enqueued_time")
    .withColumn("ingest_timestamp", current_timestamp())
)

In [0]:
query = (
    landing_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True) # (Cost Awareness!) end script if no new data in evh
    .toTable(bronze_table_name)
)